# WP4 — Partie 1 : Génération des paires (z_full_mae, z_llava)

In [4]:
pip install --upgrade pillow

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 241.4 kB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


### 1.1 Chargement MAE

In [1]:
import torch
import torch.nn.functional as F
from transformers import ViTImageProcessor, ViTMAEModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

mae_processor = ViTImageProcessor(
    size={'height': 224, 'width': 224},
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225],
)
mae_encoder = ViTMAEModel.from_pretrained('./vit-mae-large').to(DEVICE)
mae_encoder.eval()
print('MAE charge')


/home/philippelin/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cuda


/usr/local/lib/python3.10/dist-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


MAE charge


### 1.2 Chargement LLaVA — vision tower + MLP connector uniquement

On utilise `CLIPImageProcessor` directement pour eviter les problemes de compatibilite du `LlavaProcessor`. Pour la generation des paires le tokenizer n'est pas necessaire.

In [2]:
from transformers import LlavaForConditionalGeneration, CLIPImageProcessor

# Charger le modele complet
llava = LlavaForConditionalGeneration.from_pretrained(
    './llava-1.5-7b-hf',
    torch_dtype=torch.float16,
)

# Extraire uniquement vision tower et MLP connector
vision_tower  = llava.vision_tower.to(DEVICE).eval()
mlp_connector = llava.multi_modal_projector.to(DEVICE).eval()

# Liberer le LLM — inutile pour la generation des paires
del llava
torch.cuda.empty_cache()

# CLIPImageProcessor directement — evite le bug LlavaProcessor + transformers 4.37
llava_image_processor = CLIPImageProcessor.from_pretrained('./llava-1.5-7b-hf')

MAE_DIM   = 1024
LLAVA_DIM = 4096
print(f'MAE dim   : {MAE_DIM}')
print(f'LLaVA dim : {LLAVA_DIM}')
print(f'Taille image LLaVA : {llava_image_processor.size}')
print('LLaVA (vision + MLP) charge')


Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  1.86it/s]


MAE dim   : 1024
LLaVA dim : 4096
Taille image LLaVA : {'shortest_edge': 336}
LLaVA (vision + MLP) charge


### 1.3 Chargement ImageNet-100

In [3]:
from datasets import load_dataset
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

# Charger les parquet directement
ds = load_dataset('parquet', data_files={
    'train':      './imagenet100/data/train-*.parquet',
    'validation': './imagenet100/data/validation-*.parquet',
})

print(ds)
print('Colonnes :', ds['train'].column_names)
print('Exemple  :', {k: type(v) for k, v in ds['train'][0].items()})

Generating train split: 117000 examples [00:27, 4212.39 examples/s]
Generating validation split: 13000 examples [00:06, 2110.09 examples/s]


DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 117000
    })
    validation: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 13000
    })
})
Colonnes : ['image', 'label', 'text']


AttributeError: module 'PIL.Image' has no attribute 'ExifTags'

### 1.4 Génération des paires en batch

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

def generate_pairs(dataset, batch_size=32, desc=''):
    """
    Genere (z_mae, z_llava) pour chaque image.

    z_mae   : mean des 196 tokens patch MAE en full encoding (1024,)
    z_llava : mean des 576 tokens projetes par le MLP connector LLaVA (4096,)
              L2-normalise pour la coherence avec la loss cosinus

    Les images sont chargees en 336x336.
    MAE les recoit redimensionnees en 224x224 via interpolation bilineaire.
    LLaVA les recoit en 336x336 directement.
    """
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE == 'cuda'),
    )

    all_z_mae, all_z_llava, all_labels = [], [], []

    for images_336, labels in tqdm(loader, desc=desc):
        B = images_336.shape[0]

        # --- MAE : resize 336 -> 224 ---
        images_224 = F.interpolate(
            images_336, size=(224, 224), mode='bilinear', align_corners=False
        )
        mae_inputs = mae_processor(
            images=list(images_224), return_tensors='pt', do_rescale=False
        )
        mae_inputs = {k: v.to(DEVICE) for k, v in mae_inputs.items()}
        with torch.no_grad():
            out = mae_encoder(
                **mae_inputs,
                noise=torch.zeros(B, 196).to(DEVICE)
            )
        # Mean des 196 tokens patch (index 0 = CLS, on le skip)
        z_mae = out.last_hidden_state[:, 1:].mean(dim=1).cpu().float()  # (B, 1024)

        # --- LLaVA : CLIP vision tower + MLP connector ---
        llava_inputs = llava_image_processor(
            images=list(images_336), return_tensors='pt', do_rescale=False
        )
        pixel_values = llava_inputs['pixel_values'].to(DEVICE).half()
        with torch.no_grad():
            vision_out = vision_tower(pixel_values)
            # Skip le token CLS (index 0), garder les 576 tokens patch
            patch_tokens = vision_out.last_hidden_state[:, 1:]   # (B, 576, 1024)
            projected    = mlp_connector(patch_tokens)            # (B, 576, 4096)
        # Mean pooling -> representation image-level
        z_llava = projected.mean(dim=1).cpu().float()             # (B, 4096)
        z_llava = F.normalize(z_llava, dim=-1)                    # L2-normalise

        all_z_mae.append(z_mae)
        all_z_llava.append(z_llava)
        all_labels.append(labels)

    return {
        'z_mae':   torch.cat(all_z_mae),
        'z_llava': torch.cat(all_z_llava),
        'labels':  torch.cat(all_labels),
    }


### 1.5 Lancement et sauvegarde

In [ ]:
data_train = generate_pairs(dataset_train, batch_size=32, desc='Train')
data_val   = generate_pairs(dataset_val,   batch_size=32, desc='Val')

torch.save(data_train, 'wp4_pairs_train.pt')
torch.save(data_val,   'wp4_pairs_val.pt')

print('Sauvegarde OK')
print(f'  Train z_mae   : {data_train["z_mae"].shape}')    # (N, 1024)
print(f'  Train z_llava : {data_train["z_llava"].shape}')  # (N, 4096)
print(f'  Val   z_mae   : {data_val["z_mae"].shape}')

del vision_tower, mlp_connector, mae_encoder
torch.cuda.empty_cache()
print('VRAM liberee')
